In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation
import timm
from tqdm import tqdm
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from pathlib import Path

class FusionSegFormer(nn.Module):
    def __init__(self, num_classes=11, normal_class_idx=0):
        super().__init__()
        self.segformer = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0", 
            num_labels=num_classes,
            ignore_mismatched_sizes=True,
            use_safetensors=True
        )
        self.fusion_conv = nn.Conv2d(in_channels=22, out_channels=11, kernel_size=1)
        
        self.normal_class_idx = normal_class_idx 

    def forward(self, images, hints):
        outputs = self.segformer(images)
        logits = outputs.logits
        logits = F.interpolate(logits, size=images.shape[-2:], mode='bilinear', align_corners=False)

        B, C, H, W = logits.shape
        hints_spatial = hints.view(B, 11, 1, 1).expand(-1, -1, H, W)
        
        fused = torch.cat([logits, hints_spatial], dim=1)
        fused_logits = self.fusion_conv(fused)

        attention_weights = hints.view(B, C, 1, 1).clone() + 0.1 

        is_normal = (torch.argmax(hints, dim=1) == self.normal_class_idx) 

        for i in range(B):
            if is_normal[i]:
                attention_weights[i, 1:] = 0.0
                
                attention_weights[i, self.normal_class_idx] = 10.0

        final_logits = fused_logits * attention_weights

        return final_logits

In [14]:
class HintGuidedLoss(nn.Module):
    def __init__(self, lambda_exist=1.0, forgive_ratio=0.1, defect_weight=20.0):
        super().__init__()
        self.lambda_exist = lambda_exist
        self.forgive_ratio = forgive_ratio 
        self.defect_weight = defect_weight 

    def forward(self, logits, masks, hints):
        B, C, H, W = logits.shape

        weights = torch.ones(C, device=logits.device) 
        weights[1:] = self.defect_weight          

        base_loss = F.cross_entropy(logits, masks, weight=weights, ignore_index=255, reduction='none')
        
        preds = torch.argmax(logits, dim=1) 
        
        hints_spatial = hints.view(B, C, 1, 1).expand(-1, -1, H, W) 
        pred_hint_values = torch.gather(hints_spatial, 1, preds.unsqueeze(1)).squeeze(1) 
        
        forgive_mask = (masks == 0) & (pred_hint_values > 0.3) & (preds != 0)
        
        base_loss = torch.where(
            forgive_mask, 
            base_loss * self.forgive_ratio, 
            base_loss                       
        )
        
        seg_loss = base_loss.mean() 

        probs = torch.sigmoid(logits) 
        max_probs = F.adaptive_max_pool2d(probs, (1, 1)).view(B, C) 
        existence_loss = F.binary_cross_entropy(max_probs, hints)

        total_loss = seg_loss + (self.lambda_exist * existence_loss)
        return total_loss

In [15]:
class SegFusionDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=(512, 512)):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        
        self.mask_filenames = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
        
        self.image_path_map = {}
        for path in Path(image_dir).rglob("*.jpg"):
            self.image_path_map[path.name] = str(path)
            
        self.img_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_name = self.mask_filenames[idx]
        img_name = mask_name.replace('.png', '.jpg')

        img_path = self.image_path_map.get(img_name)
        
        if img_path is None:
            raise FileNotFoundError(f"'{img_name}' 파일을 찾을 수 없습니다.")

        mask_path = os.path.join(self.mask_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L") 

        image = image.resize(self.img_size, Image.BILINEAR)
        mask = mask.resize(self.img_size, Image.NEAREST)

        img_tensor = self.img_transform(image)
        mask_tensor = torch.tensor(np.array(mask), dtype=torch.long)

        return img_tensor, mask_tensor

train_image_dir = r"D:\Study\HumanStudy\Dataset\Training\01.원천데이터"
train_mask_dir = r"D:\Study\HumanStudy\Dataset\Training_Masks"

batch_size = 8

train_dataset = SegFusionDataset(train_image_dir, train_mask_dir)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

print(f" (총 {len(train_dataset)}장 / {len(train_loader)} 배치)")

 (총 98398장 / 12300 배치)


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classifier = timm.create_model('efficientnet_b0', pretrained=False, num_classes=11)
classifier.load_state_dict(torch.load('best_efficientnet_b0_multi_label.pth', weights_only=True))
classifier = classifier.to(device)
classifier.eval()
for param in classifier.parameters():
    param.requires_grad = False 

model = FusionSegFormer(num_classes=11).to(device)
criterion = HintGuidedLoss(lambda_exist=1.0, forgive_ratio=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device, dtype=torch.long)        
        images_cls = F.interpolate(images, size=(224, 224), mode='bilinear', align_corners=False)
        with torch.no_grad():
            cls_logits = classifier(images_cls)
            hints = torch.sigmoid(cls_logits) 
            hints = hints * 1.5
            hints = torch.clamp(hints, min=0.0, max=1.0)            
        optimizer.zero_grad()
        
        seg_logits = model(images, hints)
        loss = criterion(seg_logits, masks, hints)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} 평균 Loss: {epoch_loss:.4f}")
    
    save_filename = f'fusion_segformer_2_epoch_{epoch+1}.pth'
    torch.save(model.state_dict(), save_filename)
    print(f"Epoch {epoch+1} Model Saved: {save_filename} (Loss: {epoch_loss:.4f})")

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/10: 100%|█████████████████████████████████████████████████| 12300/12300 [3:00:44<00:00,  1.13it/s, loss=1.7360]


Epoch 1 평균 Loss: 1.9592
Epoch 1 Model Saved: fusion_segformer_2_epoch_1.pth (Loss: 1.9592)


Epoch 2/10: 100%|█████████████████████████████████████████████████| 12300/12300 [5:18:20<00:00,  1.55s/it, loss=1.6038]


Epoch 2 평균 Loss: 1.6790
Epoch 2 Model Saved: fusion_segformer_2_epoch_2.pth (Loss: 1.6790)


Epoch 3/10:   2%|▊                                                  | 196/12300 [08:37<8:53:05,  2.64s/it, loss=1.8180]


KeyboardInterrupt: 